### Word Embeddings

In [1]:
sample_text = [
    "I love natural language processing",
    "Transformers are advanced NLP models",
    "Machine learning is fun",
    "I enjoy learning new things"
]

tokenized_text = [sentence.lower().split() for sentence in sample_text]
tokenized_text


[['i', 'love', 'natural', 'language', 'processing'],
 ['transformers', 'are', 'advanced', 'nlp', 'models'],
 ['machine', 'learning', 'is', 'fun'],
 ['i', 'enjoy', 'learning', 'new', 'things']]

### Train Word2Vec Model


In [2]:
from gensim.models import Word2Vec

model_w2v = Word2Vec(tokenized_text, vector_size=50, window=2, min_count=1, workers=4)
model_w2v.train(tokenized_text, total_examples=len(tokenized_text), epochs=200)


(550, 3800)

### Test Word Similarity

In [3]:
model_w2v.wv.most_similar("learning")


[('language', 0.22524507343769073),
 ('are', 0.1818258911371231),
 ('processing', 0.15999214351177216),
 ('things', 0.1406877487897873),
 ('machine', 0.13613590598106384),
 ('nlp', 0.10698213428258896),
 ('love', 0.07059186697006226),
 ('i', 0.059923332184553146),
 ('transformers', 0.05391079932451248),
 ('fun', 0.02291215769946575)]

### Save the Model

In [4]:
model_w2v.save("word2vec_model.bin")


### Transformer Models

#### Load Sentiment Model

In [5]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")
sentiment("I love this new NLP module!")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998664855957031}]

### Zero-Shot Text Classification

In [6]:
classifier = pipeline("zero-shot-classification")

result = classifier(
    "Tesla is releasing a new electric vehicle",
    candidate_labels=["technology", "sports", "politics"]
)

result


No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


{'sequence': 'Tesla is releasing a new electric vehicle',
 'labels': ['technology', 'sports', 'politics'],
 'scores': [0.9835426211357117, 0.01332992222160101, 0.0031274801585823298]}

### Sentence Embeddings

#### Load Sentence Transformer Model

In [7]:
from sentence_transformers import SentenceTransformer, util

model_st = SentenceTransformer("all-MiniLM-L6-v2")


### Compare Two Sentences

In [8]:
sent1 = "I enjoy reading books"
sent2 = "I like literature"

emb1 = model_st.encode(sent1, convert_to_tensor=True)
emb2 = model_st.encode(sent2, convert_to_tensor=True)

similarity = util.cos_sim(emb1, emb2)
similarity


tensor([[0.7479]])

### Save Embedding Model Output

In [9]:
import numpy as np

np.save("sentence_embedding.npy", emb1.cpu().numpy())


### Transfer Learning (Fine-Tuning DistilBERT)

#### Load Dataset

In [10]:
from datasets import load_dataset

dataset = load_dataset("imdb", split="train[:2000]")
dataset


Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})

### Load Pretrained Model

In [11]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Training Config

In [12]:
training_args = TrainingArguments(
    output_dir="./bert_model",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=20,
)


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)


In [22]:
model.save_pretrained("./fine_tuned_bert")
